In [8]:
import pandas as pd

def read_input(file_name, lang):
    triple_df = pd.read_json('./final/'+file_name+'.json', orient='index')
    #sort by sitelink count
    triple_df_sorted = triple_df.sort_values(by='num_sitelinks',ascending=False)

    #check if entity label in all languages
    #if True, generate prompts
    triple_df_sorted_new = triple_df_sorted[triple_df_sorted['labels_count']==8] #exclude hindi
    print(lang,':',len(triple_df_sorted_new),':',len(triple_df_sorted))
    return triple_df_sorted_new

/var/folders/9y/17y9qrrx1fz1wd3fy2ywwp740000gn/T/ipykernel_70379/1642228445.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [9]:
instruction_template = {
'ar_pob':"""##Instruction
        أجب عن السؤال باللغة العربية.
        قم بالإجابة بالكلمة المفتاحية فقط. 
        ولا تقم بإرفاق أي تفسيرات.
        أعد اسم المدينة كإجابة. 
        ##Question 
        ما هو مسقط رأس {} ؟ """,
'de_pob':"""##Instruction
            Beantworte die Frage in Deutsch.
            Gib die Antwort nur in Form eines Schlüsselworts zurück.
            Füge keine Erklärungen hinzu.
            Gib den Namen der Stadt als Antwort zurück.
            ##Question
            Was ist der Geburtsort von {}?""",
'en_pob':"""##Instruction
            Answer the question in English. 
            Return only the answer keyword. 
            Do not include any explanations. 
            Return the city name as answer.  
            ##Question 
            What is the birthplace of {}?""",
'fr_pob':"""##Instruction
            Réponds à la question en Français.
            Renvoyez uniquement la réponse brute. 
            N'incluez pas d'explications. 
            Indiquez le nom de la ville comme réponse. 
            ##Question 
            Quel est le lieu de naissance de {} ?""",
'hi_pob':"""##Instruction
            प्रश्न का उत्तर प्रॉम्प्ट भाषा में दें।
            केवल उत्तर शब्द लौटाएँ।
            कोई स्पष्टीकरण शामिल न करें।
            उत्तर के रूप में शहर का नाम लौटाएँ।
            ##Question
            {} का जन्मस्थान क्या है?""",
'it_pob':"""##Instruction
            Rispondi in Italiano. 
            Rispondi soltanto con la risposta della domanda. 
            Non includere commenti o spiegazioni alla risposta. 
            Rispondi soltanto con il paese in cui è nata la persona.
            ##Question 
            Qual è il luogo di nascita di {}?""",
'pl_pob':"""##Instruction
            Odpowiedz na pytanie w języku polskim. 
            Podaj wyłącznie nazwę miejscowości. 
            Nie dodawaj żadnych wyjaśnień. 
            ##Question  
            Gdzie urodził/urodziła się {}?""",
'ru_pob':"""##Instruction
            Отвечайте на вопрос на Русском.
            Возвращайте только краткий ответ в виде ключевого слова.
            Не добавляйте никаких объяснений.
            Возвращайте название города в качестве ответа.
            ##Question
            Где родился/родилась {}?""",
'zh_pob':"""##Instruction
            用繁體中文回答給出的問題。
            只回答關鍵詞答案。
            不要給出任何解釋。
            返回答案必須是城市。
            ##Question
            {}的出生地是？""",
'ar_dob':"""##Instruction
            أجب عن السؤال باللغة العربية.
            قم بالإجابة بالكلمة المفتاحية فقط. 
            ولا تقم بإرفاق أي تفسيرات.
            قم بإرجاع الإجابات بصيغة ”YYYYY-MM-DD“. 
            ##Question
            ما هو تاريخ ميلاد {} ؟""",
'de_dob':"""##Instruction
            Beantworte die Frage in Deutsch.
            Gib die Antwort nur in Form eines Schlüsselworts zurück.
            Füge keine Erklärungen hinzu.
            Gib Antworten im Format ‘YYYY-MM-DD’ zurück.
            ##Question
            Was ist das Geburtsdatum von {}?""",
'en_dob':"""##Instruction
            Answer the question in English. 
            Return only the answer keyword. 
            Do not include any explanations. 
            Return answers in 'YYYY-MM-DD' format.  
            ##Question 
            What is the birth date of {}?""",
'fr_dob':"""##Instruction
            Réponds à la question en Français. 
            Renvoyez uniquement la réponse brute. 
            N'incluez pas d'explications. 
            Retournez les réponses au format 'YYYY-MM-DD'. 
            ##Question 
            Quelle est la date de naissance de {}?""",
'hi_dob':"""##Instruction
            प्रश्न का उत्तर प्रॉम्प्ट भाषा में दें।
            केवल उत्तर शब्द लौटाएँ।
            कोई स्पष्टीकरण शामिल न करें।
            उत्तर 'YYYY-MM-DD' प्रारूप में लौटाएँ।
            ##Question
            {} की जन्म तिथि क्या है?""",
'it_dob':"""##Instruction
            Rispondi in Italiano. 
            Rispondi soltanto con la risposta della domanda. 
            Non includere commenti o spiegazioni alla risposta. 
            Ritorna la risposta nel seguente formato: 'YYYY-MM-DD'. 
            ##Domanda
            Qual è la data di nascita di {}?""",
'pl_dob':"""##Instruction 
            Odpowiedz na pytanie w języku polskim. 
            Odpowiedz na pytanie podając wyłącznie datę w formacie „RRRR-MM-DD". 
            ##Question 
            # Kiedy urodził/urodziła się {}?""",
'ru_dob':"""##Instruction
            Отвечайте на вопрос на Русском.
            Возвращайте только краткий ответ в виде ключевого слова.
            Не добавляйте никаких объяснений.
            Возвращайте ответы в формате ‘YYYY-MM-DD’.
            ##Question
            Какова дата рождения {}?""",
'zh_dob':"""##Instruction
            用繁體中文回答給出的問題。
            只回答關鍵詞答案。
            不要給出任何解釋。
            返回答案遵循下面的形式’YYYY-MM-DD’。
            ##Question
            {}的生日是？""",
'ar_occ':"""##Instruction
                أجب عن السؤال باللغة العربية.
                قم بالإجابة بالكلمة المفتاحية فقط. 
                الإجابة لابد أن تكون على هذا الشكل: [الإجابة 1، الإجابة 2، ...].
                ولا تقم بإرفاق أي تفسيرات.
                ##Question 
                ما هي وظيفة {} ؟""",
'de_occ':"""##Instruction
                Beantworte die Frage in Deutsch.
                Gib die Antwort nur in Form eines Schlüsselworts zurück.
                Die Antwort muss diesem Format entsprechen: [Antwort1, Antwort2, ...]
                Füge keine Erklärungen hinzu.
                ##Question
                Was ist der Beruf von {}?""",
'en_occ':"""##Instruction
                Answer the question in English. 
                Return only the answer keyword.
                Answer must follow this format: [answer1, answer2,...] 
                Do not include any explanations.  
                ##Question 
                What is the occupation of {}?""",
'fr_occ':"""##Instruction
                Réponds à la question en Français.
                Ne renvoyez que le mot-clé de la réponse.
                La réponse doit impérativement respecter le format suivant : [answer1, answer2,...]
                Aucune explication ne doit être incluse.
                ##Question 
                Quelle est la profession de {} ?""",
'hi_occ':"""##Instruction
                प्रश्न का उत्तर प्रॉम्प्ट भाषा में दें।
                केवल उत्तर शब्द  लौटाएँ।
                कोई स्पष्टीकरण शामिल न करें।
                ##Question
                {} का व्यवसाय क्या है?""",
'it_occ':"""##Istruzione
                Rispondi in Italiano. 
                Rispondi soltanto con la parola chiave richiesta. 
                La risposta deve seguire questo formato: [answer1, answer2,...]
                Non includere spiegazioni. 
                ##Question 
                Qual è la occupazione (o il lavoro) di {}?""",
'pl_occ':"""##Instruction
                Odpowiedz na pytanie w języku polskim.
                Podaj tylko odpowiedź.
                Odpowiedź musi mieć format: [odpowiedź1, odpowiedź2,...]
                Nie dodawaj żadnych wyjaśnień.
                ##Question 
                Kim z zawodu jest {}?""",
'ru_occ':"""##Instruction
                Отвечайте на вопрос на Русском.
                Возвращайте только краткий ответ в виде ключевого слова.
                Ответ должен соответствовать формату: [ответ1, ответ2, …]
                Не добавляйте никаких объяснений.
                ##Question
                Какова профессия {}?""",
'zh_occ':"""##Instruction
                用繁體中文回答給出的問題。
                只回答關鍵詞答案。
                返回答案遵循下面的形式: [‘答案1’, ‘答案2’，...]。
                不要給出任何解釋。
                ##Question
                {}的職業是？""",
'ar_ctry':"""##Instruction
                أجب عن السؤال باللغة العربية.
                قم بالإجابة بالكلمة المفتاحية فقط. 
                الإجابة لابد أن تكون على هذا الشكل: [الإجابة 1، الإجابة 2، ...].
                ولا تقم بإرفاق أي تفسيرات.
                ##Question 
                ما هو البلد الذي ينتمي إليه {} ؟""",
'de_ctry':"""##Instruction
                Beantworte die Frage in Deutsch.
                Gib die Antwort nur in Form eines Schlüsselworts zurück.
                Die Antwort muss diesem Format entsprechen: [Antwort1, Antwort2, ...]
                Füge keine Erklärungen hinzu.
                ##Question
                Was ist die Staatsangehörigkeit von {}?""",
'en_ctry':"""##Instruction
                Answer the question in English. 
                Return only the answer keyword.
                Answer must follow this format: [answer1, answer2,...] 
                Do not include any explanations. 
                ##Question 
                What is the country of citizenship of {}?""",
'fr_ctry':"""##Instruction
                Réponds à la question en Français.
                Ne renvoyez que le mot-clé de la réponse.
                La réponse doit impérativement respecter le format suivant : [answer1, answer2,...]
                Aucune explication ne doit être incluse.
                ##Question 
                Quel est le pays de citoyenneté de {} ?""",
'hi_ctry':"""##Instruction
                प्रश्न का उत्तर प्रॉम्प्ट भाषा में दें।
                केवल उत्तर शब्द लौटाएँ।
                कोई स्पष्टीकरण शामिल न करें।
                ##Question
                {} की नागरिकता किस देश की है?""",
'it_ctry':"""##Instruction
                Rispondi in Italiano. 
                Rispondi soltanto con la parola chiave richiesta. 
                La risposta deve seguire solo questo formato: [answer1, answer2,...]
                Non includere spiegazioni. 
                ##Question 
                Che cittadinanza ha {}?""",
'pl_ctry':"""##Instruction
                Odpowiedz na pytanie w języku polskim.
                Podaj nazwę kraju/krajów bez dalszego wyjaśnienia.
                Odpowiedź musi mieć format: [odpowiedź1, odpowiedź2,...] 
                ##Question 
                Jakiego państwa obywatelem jest {}?""",
'ru_ctry':"""##Instruction
                Отвечайте на вопрос на Русском.
                Возвращайте только краткий ответ в виде ключевого слова.
                Ответ должен соответствовать формату: [ответ1, ответ2, …]
                Не добавляйте никаких объяснений.
                ##Question
                Какое гражданство у {}?""",
'zh_ctry':"""##Instruction
               用繁體中文回答給出的問題。
               只回答關鍵詞答案。
               返回答案遵循下面的形式: [‘答案1’, ‘答案2’，...]。
               不要給出任何解釋。
               ##Question
               {}的國籍是？"""
}

In [10]:
def prompt_generate_pob(ent, df, lang):
    gt, prompt='',''
    ent_pob = df['P19']['values']
    ent_pob = ent_pob[0]['labels']        
    if lang in ent_pob.keys():
        gt = ent_pob[lang]
        prompt = instruction_template.get(lang+'_pob').format(ent)
    return gt, prompt
    

In [11]:
def prompt_generate_dob(ent, df, lang):
    gt, prompt='',''
    ent_dob = df['P569']['values']
    gt = ent_dob[0]['qid']
    prompt = instruction_template.get(lang+'_dob').format(ent)
    return gt, prompt

In [12]:
def prompt_generate_occ(ent, df, lang):
    gt = []
    prompt=''
    if not pd.isnull(df['P106']):
        #print(df['P106'])
        ent_occupation = df['P106']['values']
        for job in ent_occupation:
            occupation = job['labels']
            if lang in occupation.keys():
                gt.append(occupation[lang])
                prompt = instruction_template.get(lang+'_occ').format(ent)
    return gt, prompt

In [13]:
def prompt_generate_country(ent, df, lang):
    gt = []
    prompt=''
    if not pd.isnull(df['P27']):
        ent_citizen_country = df['P27']['values']   
        for country in ent_citizen_country:
            citizen_country = country['labels']
            if lang in citizen_country.keys():
                gt.append(citizen_country[lang])
                prompt = instruction_template.get(lang+'_ctry').format(ent)
    return gt, prompt

In [14]:
def generate_prompts(triple_df, ent_id_lst, lang_lst, prop_lst):
    prompt = []
    for ent in ent_id_lst:
        ent_names = triple_df.loc[ent,'labels']
        for lang in lang_lst:
            #check if ent label exists in lang
            if lang in ent_names.keys():
                for prop in prop_lst:
                    if not pd.isnull(triple_df.loc[ent,'P19']):
                        pob_gt, pob_prompt = prompt_generate_pob(ent_names[lang],triple_df.loc[ent],lang)
                    if not pd.isnull(triple_df.loc[ent,'P569']):
                        dob_gt, dob_prompt = prompt_generate_dob(ent_names[lang],triple_df.loc[ent],lang)
                    if not pd.isnull(triple_df.loc[ent,'P106']):
                        occ_gt, occ_prompt = prompt_generate_occ(ent_names[lang],triple_df.loc[ent],lang)
                    if not pd.isnull(triple_df.loc[ent,'P27']):
                        country_gt, country_prompt = prompt_generate_country(ent_names[lang],triple_df.loc[ent],lang)
                prompt.append({'ent_ID':ent, 'lang':lang, 'pob_ground_truth':pob_gt,'pob_prompt':pob_prompt,'dob_ground_truth':dob_gt,'dob_prompt':dob_prompt,'occ_ground_truth':occ_gt,'occ_prompt':occ_prompt,'country_ground_truth':country_gt,'country_prompt':country_prompt,'num_sitelinks':triple_df.loc[ent,'num_sitelinks']})      
    return prompt

In [ ]:
file_names = {'ar':'Arabic_merged','zh':'Chinese_merged','en':'English_merged','fr':'French_merged','de':'German_merged','ru':'Russian_merged','it':'Italian_merged','pl':'Polish_merged'}
lang_lst = ['ar', 'de', 'en', 'fr', 'ru', 'zh', 'it', 'pl']
prop_lst = ['P19','P569','P106','P27']

result_df = pd.DataFrame(columns=['ent_ID','pob_ground_truth','pob_prompt','dob_ground_truth','dob_prompt','occ_ground_truth','occ_prompt','country_ground_truth','country_prompt','num_sitelinks'])

for lang, file in file_names.items():
    ground_truth_df = read_input(file, lang)
    ent_id_lst = ground_truth_df.index
    prompts = generate_prompts(ground_truth_df, ent_id_lst, lang_lst, prop_lst)
    
    result_df = pd.DataFrame(prompts)
    result_df = result_df.sort_values(['num_sitelinks'], ascending=False)
    result_df.to_json('./prompt/prompt_'+lang+'.json',indent=4, orient='records', force_ascii=False)


ar : 274 : 388
zh : 115 : 177
en : 3476 : 4188
fr : 374 : 491
de : 350 : 521
ru : 308 : 508
it : 436 : 653
pl : 137 : 231


In [16]:
result_df

,ent_ID,lang,pob_ground_truth,pob_prompt,dob_ground_truth,dob_prompt,occ_ground_truth,occ_prompt,country_ground_truth,country_prompt,num_sitelinks
0,Q51552,ar,باريس,##Instruction\n أجب عن السؤال باللغة ال...,1933-08-18T00:00:00Z,##Instruction\n أجب عن السؤال باللغ...,"[ممثل أفلام, مخرج أفلام, كاتب سيناريو, منتج أف...",##Instruction\n أجب عن السؤال ب...,"[فرنسا, بولندا]",##Instruction\n أجب عن السؤال ب...,92
2,Q51552,en,Paris,##Instruction\n Answer the question...,1933-08-18T00:00:00Z,##Instruction\n Answer the question...,"[film actor, film director, screenwriter, film...",##Instruction\n Answer the ques...,"[France, Poland]",##Instruction\n Answer the ques...,92
3,Q51552,fr,Paris,##Instruction\n Réponds à la questi...,1933-08-18T00:00:00Z,##Instruction\n Réponds à la questi...,"[acteur ou actrice de cinéma, réalisateur ou r...",##Instruction\n Réponds à la qu...,"[France, Pologne]",##Instruction\n Réponds à la qu...,92
4,Q51552,ru,Париж,##Instruction\n Отвечайте на вопрос...,1933-08-18T00:00:00Z,##Instruction\n Отвечайте на вопрос...,"[киноактёр, кинорежиссёр, сценарист, кинопродю...",##Instruction\n Отвечайте на во...,"[Франция, Польша]",##Instruction\n Отвечайте на во...,92
5,Q51552,zh,巴黎,##Instruction\n 用繁體中文回答給出的問題。\n ...,1933-08-18T00:00:00Z,##Instruction\n 用繁體中文回答給出的問題。\n ...,"[電影演員, 電影導演, 編劇, 电影制片人, 戏剧导演, 演員, 導演, 作家, 性格演員]",##Instruction\n 用繁體中文回答給出的問題。\n...,"[法國, 波蘭]",##Instruction\n 用繁體中文回答給出的問題。\n ...,92
...,...,...,...,...,...,...,...,...,...,...,...
770,Q455656,en,Roczyny,##Instruction\n Answer the question...,1925-10-19T00:00:00Z,##Instruction\n Answer the question...,"[military officer, politician]",##Instruction\n Answer the ques...,[Poland],##Instruction\n Answer the ques...,28
771,Q455656,fr,Roczyny,##Instruction\n Réponds à la questi...,1925-10-19T00:00:00Z,##Instruction\n Réponds à la questi...,"[officier, personnalité politique]",##Instruction\n Réponds à la qu...,[Pologne],##Instruction\n Réponds à la qu...,28
772,Q455656,ru,,,1925-10-19T00:00:00Z,##Instruction\n Отвечайте на вопрос...,"[офицер, политик]",##Instruction\n Отвечайте на во...,[Польша],##Instruction\n Отвечайте на во...,28
773,Q455656,zh,罗辛尼,##Instruction\n 用繁體中文回答給出的問題。\n ...,1925-10-19T00:00:00Z,##Instruction\n 用繁體中文回答給出的問題。\n ...,"[軍官, 政治人物]",##Instruction\n 用繁體中文回答給出的問題。\n...,[波蘭],##Instruction\n 用繁體中文回答給出的問題。\n ...,28
